In [36]:
# for importing utils from python scripts in parent directories

import sys
sys.path.append('../utils')
sys.path.append('..')

In [37]:
# suppress TensorFlow warnings and logs

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0 = all, 1 = info, 2 = warning, 3 = error

import warnings
warnings.filterwarnings('ignore')

# If you use logging, also suppress it:
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

In [38]:
import tensorflow as tf


gpus = tf.config.experimental.list_physical_devices('GPU')
print("GPUs available:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Physical devices cannot be modified after being initialized


## Defining the image processing function

In [39]:
def preprocess_images(df, preproc_fn):
    X = []
    y = []
    for idx, row in df.iterrows():
        img_path = row['image_path']
        label = row['label']
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Could not read {img_path}")
            continue
        processed = preproc_fn(img)
        X.append(processed)
        y.append(label)
    X = np.stack(X)
    y = np.stack(y)
    return X, y

## Get the training and testing data

In [40]:
import pandas as pd

train_df = pd.read_csv('../data/train_split.csv')
test_df = pd.read_csv('../data/test_split.csv')

print(train_df.columns)
print(train_df.head())

print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

Index(['image_path', 'label'], dtype='object')
                                   image_path label
0  ../data/augmented_images/22580576_aug1.png     5
1  ../data/augmented_images/20587518_aug1.png     2
2  ../data/augmented_images/50994191_orig.png     1
3  ../data/augmented_images/24055274_aug1.png     5
4  ../data/augmented_images/22427705_aug0.png     5
Train set: 1230 samples
Test set: 410 samples


In [41]:
# Map string labels to integer values for model training
label_mapping = {
    '1': 0,
    '2': 1,
    '3': 2,
    '4a': 3,
    '4b': 4,
    '4c': 5,
    '5': 6,
    '6': 7
}

train_df['label'] = train_df['label'].map(label_mapping)
test_df['label'] = test_df['label'].map(label_mapping)

print('Label mapping applied. Unique train labels:', train_df['label'].unique())
print('Label mapping applied. Unique test labels:', test_df['label'].unique())

Label mapping applied. Unique train labels: [6 1 0 5 2 7 3 4]
Label mapping applied. Unique test labels: [1 6 0 5 2 4 3 7]


## Running the Models

### Define the model's running with the preprocessing function

In [42]:
import numpy as np
from keras.utils import to_categorical
from keras import backend as K

def run_model_with_preprocessing(
    train_df, test_df, preproc_fn, model_name, model_fn, num_classes=8, batch_size=8, epochs=15
):
    X_train, y_train = preprocess_images(train_df, preproc_fn)
    X_test, y_test = preprocess_images(test_df, preproc_fn)
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0

    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)

    if model_name.lower() in ['custom cnn']:
        input_shape = X_train.shape[1:]
        X_tr, X_te = X_train, X_test
    else:
        X_train_3ch = np.repeat(X_train, 3, axis=-1)
        X_test_3ch = np.repeat(X_test, 3, axis=-1)
        input_shape = X_train_3ch.shape[1:]
        X_tr, X_te = X_train_3ch, X_test_3ch
        batch_size = 4

    model = model_fn(input_shape=input_shape, num_classes=num_classes)
    history = model.fit(
        X_tr, y_train,
        validation_data=(X_te, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=2
    )
    val_accuracies = history.history['val_accuracy']
    best_epoch = int(np.argmax(val_accuracies)) + 1
    best_val_acc = float(np.max(val_accuracies))

    K.clear_session()
    del model
    import gc; gc.collect()

    return best_val_acc, best_epoch, history.history

### Actually running the models

In [43]:
import os

def check_if_model_exists(preproc_id, model_name, combo_str):
    history_file = f"../data/history_{preproc_id}_{model_name}_{combo_str.replace(', ', '_').replace('=', '-')}.csv"
    return os.path.exists(history_file)

In [44]:
import itertools
import cv2
import numpy as np
from utils.preprocessing import preprocessing_methods
from utils.models import MODEL_BUILDERS
import time
import os

batch_size = 8
run_skip = True # if True, will skip already ran models

for preproc_id, preproc_info in preprocessing_methods.items():
    param_names = list(preproc_info['params'].keys())
    param_values = [preproc_info['params'][k] for k in param_names]
    for param_combo in itertools.product(*param_values):
        param_dict = dict(zip(param_names, param_combo))
        def preproc_fn(img, func=preproc_info['func'], params=param_dict):
            return func(img, **params)
        combo_str = ', '.join([f"{k}={v}" for k, v in param_dict.items()])
        print(f"\n=== Preprocessing: {preproc_id} ({combo_str}) ===")
        for model_name, model_fn in MODEL_BUILDERS.items():
            print(f"--> Current Model: {model_name} <--")
            if run_skip and check_if_model_exists(preproc_id, model_name, combo_str):
                print(f"Skipping {preproc_id} [{model_name} - {combo_str}] as it has already been run.")
                continue
            model_time = time.time()
            best_val_acc, best_epoch, history_dict = run_model_with_preprocessing(
                train_df, test_df, preproc_fn, model_name, model_fn, num_classes=8, batch_size=batch_size, epochs=15
            )
            hist_df = pd.DataFrame(history_dict)
            hist_df.to_csv(
                f"../data/history_{preproc_id}_{model_name}_{combo_str.replace(', ', '_').replace('=', '-')}.csv",
                index=False
            )
            
            end_model_time = time.time()
            print(f"Best val accuracy for {preproc_id} [{model_name} - {combo_str}]: {best_val_acc:.4f} at epoch {best_epoch}")
            
            elapsed = end_model_time - model_time
            h = int(elapsed // 3600)
            m = int((elapsed % 3600) // 60)
            s = int(elapsed % 60)
            parts = []
            if h > 0:
                parts.append(f"{h}h")
            if m > 0:
                parts.append(f"{m}min")
            if s > 0 or not parts:
                parts.append(f"{s}sec")
            print(f"Time taken for {preproc_id} [{model_name}]: {' '.join(parts)}\n")


=== Preprocessing: denoise (kernel_size=(3, 3), sigma=0) ===
--> Current Model: custom cnn <--
Skipping denoise [custom cnn - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: resnet <--
Skipping denoise [resnet - kernel_size=(3, 3), sigma=0] as it has already been run.
--> Current Model: densenet <--
Skipping denoise [densenet - kernel_size=(3, 3), sigma=0] as it has already been run.

=== Preprocessing: denoise (kernel_size=(3, 3), sigma=1) ===
--> Current Model: custom cnn <--
Skipping denoise [custom cnn - kernel_size=(3, 3), sigma=1] as it has already been run.
--> Current Model: resnet <--
Skipping denoise [resnet - kernel_size=(3, 3), sigma=1] as it has already been run.
--> Current Model: densenet <--
Skipping denoise [densenet - kernel_size=(3, 3), sigma=1] as it has already been run.

=== Preprocessing: denoise (kernel_size=(3, 3), sigma=2) ===
--> Current Model: custom cnn <--
Skipping denoise [custom cnn - kernel_size=(3, 3), sigma=2] as it has al

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [ ]:
# save the results df to a csv file
results_df.to_csv('../data/preprocessing_model_results.csv', index=False)
print("Results saved to ../data/preprocessing_model_results.csv")